# 00 — Landuse Join Rebuild

**SDI Tehran Pipeline · Notebook 0 (branch: `experiment/landuse-join-fix`)**

## Purpose
Rebuild the `landuse` attribute from scratch using a corrected spatial join, and merge the result into a fixed copy of the training set. This replaces the original manual QGIS join (snap-to-grid centroid + ~2m buffer), which was diagnosed to systematically under-assign labels: dense areas where many small structures share one dissolved/merged landuse polygon only produced a single centroid, so only one segment near that centroid got labeled — every other segment geometrically inside the same polygon was left NULL. This produced a 33.2% null rate in `landuse` (vs. ~4% and 0% for `dilap_index` and `policy_era` respectively, which use a different, unaffected source/method) and is scoped to `landuse` only.

## Methodology note — join method (polygon-buffer majority vote)
Segments are `MultiLineString` geometries (street/axial centerlines); landuse polygons represent parcels/buildings set back from those centerlines. A raw line-vs-polygon `intersects` test misses most segments — a line down a street rarely literally touches a parcel. This notebook instead identifies, for every landuse polygon, **which segments plausibly border it** by buffering the polygon itself (not a centroid) by a fixed distance spanning the typical parcel-edge-to-street-centerline gap, then finding all segments that fall inside that buffered polygon. Each segment accumulates one "vote" per landuse polygon it falls within the buffer of; a segment bordering several same-class parcels naturally accumulates more votes for that class. Each segment is assigned the **majority-vote class** among its candidate polygons.

This differs from a plain nearest-centroid approach in an important way: a segment gets credit from every polygon it plausibly borders, not just the single nearest one, so dense areas where many small structures share a class don't get out-voted by one large adjacent polygon of a different class.

**Fallback:** any segment that falls outside every buffered polygon (e.g. sits further from all parcels than the buffer distance reaches) is assigned via nearest-centroid instead, guaranteeing full coverage within the footprint — no segment inside the surveyed area is left NULL purely because the buffer didn't reach it.

## Methodology note — footprint scoping
The footprint restriction (Section 5) must guarantee inclusion of every segment that appears in `training_set.csv`, regardless of a bounding-box pre-filter — a segment can legitimately sit outside a tight bbox-plus-margin around `landuse.gpkg`'s extent while still being close enough to a landuse polygon for the buffer/vote logic to resolve it correctly. An earlier version of this notebook used the bbox as a hard cutoff, which silently dropped ~39% of `training_set` before the join ever ran, producing a *worse* null rate (39.27%) than the original bug (33.19%) despite the join logic itself working correctly. `training_set.csv` is therefore loaded early (Section 4) so its segment IDs can be force-included in the footprint, independent of the bbox.

## Methodology note — source data cleaning
Two data-quality issues in `landuse.gpkg` were identified during diagnosis and are handled explicitly here, not silently:

1. **Unparseable geometry (31 of 657,470 polygons):** a small number of polygons have literally invalid WKB geometry (non-closed linear rings), which raises `GEOSException` on load. These are loaded with `on_invalid="warn"` (sets unparseable geometries to `None` rather than failing the whole read) and then dropped. Spread across classes 1/2/3/4/9 — not concentrated in any one class, trivial in scale.
2. **`NEW_LU == 0` (666 of 657,470 polygons, 0.1%) excluded as a digitization artifact, not a real landuse category.** Diagnosed via overlap analysis against neighboring polygons: 10.5% are near-duplicates (≥90% overlap with another class, mostly class 10/Unidentified and class 1/Residential), 34.8% are boundary slivers (<10% overlap — seam noise between adjacent digitized polygons), 53.2% partial/ambiguous overlap, 1.5% standalone. None represent a coherent real-world landuse category — confirmed by both this quantitative overlap analysis and manual visual inspection in QGIS. This exclusion is applied in-memory only, before the join; `landuse.gpkg` itself is left untouched on disk.
3. **776 topologically invalid (but parseable) polygons** — self-intersecting rings that pass WKB parsing but raise `GEOSException: TopologyException` during geometric operations. Repaired in-memory via `shapely.make_valid()`, run after class-0 exclusion; distinct from issue 1 (parse failure vs. topology failure require different handling).

With class 0 excluded, the remaining raw values 1–10 map cleanly 1:1 onto the 10-label landuse dictionary (raw 1 = Residential ... raw 10 = Unidentified).

## CRS handling
`segments.geojson` and `landuse.gpkg` may not share a CRS. **Segments are reprojected into landuse's native CRS (EPSG:32639, a projected/metric CRS)** — not the other way around — so that buffering and distance operations are computed in meters rather than degrees, where distance/area figures would be distorted and not comparable across latitudes.

## Inputs
- `segments.geojson` — segment geometries, all segments, keyed by `Depthmap_R`
- `landuse.gpkg` — landuse polygons, field `NEW_LU`
- `training_set.csv` — existing training set (flawed `landuse` column to be replaced)

## Outputs (isolated to `experiments/landuse-join-fix/`)
- `landuse_joined.csv` → full segments-layer lookup (`Depthmap_R`, `landuse`), diagnostic/audit artifact
- `training_set_landuse_fixed.csv` → training set with corrected `landuse` column, ready for NB01 onward

## 1 · Setup and load inputs

In [1]:
import sys, os
sys.path.insert(0, '..')
import config
import geopandas as gpd
import pandas as pd

SEG_ID_COL = config.SEG_ID_COL       # "Depthmap_R"
LANDUSE_FIELD = "NEW_LU"

segments = gpd.read_file(config.SEGMENTS_GEOJSON)

# on_invalid="warn" avoids a hard GEOSException crash on the ~31 known
# corrupt-geometry rows in landuse.gpkg; unparseable geometries load as None
# and are dropped explicitly in the next cell. This geopandas/shapely version
# only supports on_invalid in {"warn","raise","ignore"} — no "fix" option.
landuse = gpd.read_file(config.LANDUSE_GPKG, on_invalid="warn")

# Loaded early (not just before the final merge) so training_set segment IDs
# can be force-included in the footprint restriction in Section 5 — see
# methodology note above on footprint scoping.
training_set = pd.read_csv(config.TRAINING_SET_CSV)

print(f"Segments: {len(segments)} rows, CRS: {segments.crs}")
print(f"Landuse:  {len(landuse)} rows (pre-cleaning), CRS: {landuse.crs}")
print(f"Training set: {len(training_set)} rows")
print(f"Landuse field '{LANDUSE_FIELD}' present: {LANDUSE_FIELD in landuse.columns}")
print(f"Segment ID column '{SEG_ID_COL}' present in segments: {SEG_ID_COL in segments.columns}")

c:\Users\sepehr\AppData\Local\Programs\Python\Python39\lib\site-packages\shapely\io.py:320: Warning: Invalid WKB: geometry is returned as None. IllegalArgumentException: Points of LinearRing do not form a closed linestring
  return lib.from_wkb(geometry, invalid_handler, **kwargs)


Segments: 427100 rows, CRS: EPSG:4326
Landuse:  657470 rows (pre-cleaning), CRS: EPSG:32639
Training set: 147414 rows
Landuse field 'NEW_LU' present: True
Segment ID column 'Depthmap_R' present in segments: True


## 2 · Clean source landuse layer

Exclusions applied in-memory, `landuse.gpkg` on disk is untouched:
1. Drop rows with unparseable geometry (loaded as `None` due to `on_invalid="warn"`)
2. Drop `NEW_LU == 0` — confirmed digitization artifact, not a real landuse category (see notebook intro)

In [2]:
null_geom_count = landuse.geometry.isna().sum()
print(f"Rows with unparseable geometry: {null_geom_count}")
if null_geom_count > 0:
    print(landuse[landuse.geometry.isna()][LANDUSE_FIELD].value_counts())
landuse = landuse[landuse.geometry.notna()].copy()

class0_count = (landuse[LANDUSE_FIELD] == 0).sum()
print(f"\nRows with NEW_LU == 0 (digitization artifact, excluding): {class0_count}")
landuse = landuse[landuse[LANDUSE_FIELD] != 0].copy()

print(f"\nLanduse polygons after cleaning: {len(landuse)}")
print(f"Remaining class distribution:")
print(landuse[LANDUSE_FIELD].value_counts().sort_index())

Rows with unparseable geometry: 31
NEW_LU
1    23
2     4
9     2
3     1
4     1
Name: count, dtype: int64

Rows with NEW_LU == 0 (digitization artifact, excluding): 666

Landuse polygons after cleaning: 656773
Remaining class distribution:
NEW_LU
1     525407
2       8849
3      17557
4      13050
5      67078
6       4595
7        249
8       1715
9      13485
10      4788
Name: count, dtype: int64


## 2.1 · Repair topologically invalid geometries

Distinct from the unparseable-WKB rows dropped above: these 776 polygons parse successfully but are self-intersecting, which crashes geometric operations with `GEOSException: TopologyException`. Repaired in-memory via `make_valid()`, run after class-0 exclusion since class-0 rows are dropped, not repaired.

In [3]:
from shapely import make_valid

invalid_mask = ~landuse.geometry.is_valid
print(f"Topologically invalid geometries: {invalid_mask.sum()}")

landuse["geometry"] = landuse.geometry.apply(lambda g: make_valid(g) if not g.is_valid else g)

print(f"Still invalid after repair: {(~landuse.geometry.is_valid).sum()}")
print(landuse.geometry.type.value_counts())

Topologically invalid geometries: 776
Still invalid after repair: 0
MultiPolygon          656195
GeometryCollection       500
Polygon                   77
LineString                 1
Name: count, dtype: int64


## 3 · Align CRS

Segments are reprojected into landuse's native CRS (EPSG:32639, metric) — not the reverse — so all downstream buffering and distance operations happen in meters.

In [4]:
if segments.crs != landuse.crs:
    segments_proj = segments.to_crs(landuse.crs)   # segments -> EPSG:32639
    print(f"Reprojected segments from {segments.crs} to {segments_proj.crs}")
else:
    segments_proj = segments.copy()
    print(f"CRS already aligned: {segments.crs}")

Reprojected segments from EPSG:4326 to EPSG:32639


## 4 · Restrict to segments within the landuse footprint (training-set segments always included)

`landuse.gpkg` only covers formal, surveyed Tehran — a small subset of the full segments layer (which spans the whole city including the periphery). A bounding-box filter (`.cx[]`) is used as a cheap pre-filter for the large prediction-only segment population, purely for performance — not as a precision cut (a raw `intersects` filter here would reintroduce the same street-vs-parcel miss problem this notebook exists to fix; the real spatial logic happens in Section 5).

**Every segment present in `training_set.csv` is force-included regardless of the bbox.** An earlier version of this notebook used the bbox as a hard cutoff and silently dropped ~39% of `training_set` — segments that sit outside the bbox+margin but are still close enough to a landuse polygon for Section 5's buffer/vote logic to resolve correctly. That produced a *worse* null rate (39.27%) than the original bug (33.19%), despite the join logic itself working fine — a pure scoping bug, not a join bug.

In [5]:
train_ids = set(training_set[SEG_ID_COL])

minx, miny, maxx, maxy = landuse.total_bounds
margin = 50  # meters — cheap pre-filter for prediction-only segments only
bbox_index = segments_proj.cx[minx-margin:maxx+margin, miny-margin:maxy+margin].index

is_train = segments_proj[SEG_ID_COL].isin(train_ids)
is_in_bbox = segments_proj.index.isin(bbox_index)

segments_in_footprint = segments_proj[is_train | is_in_bbox].copy()

print(f"Total segments: {len(segments_proj)}")
print(f"Training-set segments (always included): {is_train.sum()}")
print(f"Additional segments within bbox+{margin}m (prediction-only): {(is_in_bbox & ~is_train).sum()}")
print(f"Segments within footprint (union): {len(segments_in_footprint)}")

n_train_outside_bbox = (is_train & ~is_in_bbox).sum()
print(f"\nTraining-set segments that were outside the bbox (now force-included): {n_train_outside_bbox}")

Total segments: 427100
Training-set segments (always included): 147414
Additional segments within bbox+50m (prediction-only): 77833
Segments within footprint (union): 225247

Training-set segments that were outside the bbox (now force-included): 57885


## 5 · Buffer landuse polygons and find bordering segments (majority vote)

For every landuse polygon, buffer it outward by `POLY_BUFFER_M` — a distance meant to span the gap from a parcel's edge, across any sidewalk/setback, to the street centerline where the segment geometry actually runs. Any segment falling inside that buffered polygon is treated as "bordering" it and casts one vote for that polygon's class. A segment bordering multiple polygons (e.g. several small same-class parcels) accumulates multiple votes; each segment is ultimately assigned its majority-class.

**`POLY_BUFFER_M` is a judgment call, not derived from data** — it should roughly match typical street width plus setback in the study area. Tune and re-run this cell if the resulting match/vote statistics look off.

In [6]:
POLY_BUFFER_M = 10  # tune based on typical street width + setback in the study area

landuse_buffered = landuse.copy()
landuse_buffered["geometry"] = landuse_buffered.geometry.buffer(POLY_BUFFER_M)

joined = gpd.sjoin(
    segments_in_footprint, landuse_buffered[[LANDUSE_FIELD, "geometry"]], predicate="intersects"
)

vote_counts = joined.groupby([SEG_ID_COL, LANDUSE_FIELD]).size().reset_index(name="votes")
n_matched_segments = joined[SEG_ID_COL].nunique()
n_multi_class = vote_counts.groupby(SEG_ID_COL)[LANDUSE_FIELD].nunique()
n_multi_class = (n_multi_class > 1).sum()

print(f"Segments matched via buffered-polygon vote: {n_matched_segments}")
print(f"Segments with votes split across >1 class: {n_multi_class}")

Segments matched via buffered-polygon vote: 138950
Segments with votes split across >1 class: 76224


## 6 · Resolve majority vote, with nearest-centroid fallback

Each matched segment is assigned the class with the most votes (ties broken arbitrarily — low priority to resolve further given expected tie frequency). Any segment that fell outside every buffered polygon in Section 5 (i.e. no candidate found) falls back to nearest-centroid assignment, guaranteeing every footprint segment gets a label.

**Note on fallback class skew:** the nearest-centroid fallback (10m buffer) is used more often for classes 9 (Vacant) and 10 (Unidentified) than for other classes — these parcel types tend to be larger and more sparsely distributed, so segments near them are less likely to fall within a 10m buffer of the polygon edge. This is a known, expected property of parcel geometry for these two classes, not a join defect.

In [7]:
majority = (
    vote_counts.sort_values("votes", ascending=False)
    .drop_duplicates(subset=SEG_ID_COL)[[SEG_ID_COL, LANDUSE_FIELD]]
    .rename(columns={LANDUSE_FIELD: "landuse"})
)

landuse_centroids = landuse.copy()
landuse_centroids["geometry"] = landuse_centroids.geometry.centroid

matched_ids = set(majority[SEG_ID_COL])
unmatched = segments_in_footprint[~segments_in_footprint[SEG_ID_COL].isin(matched_ids)]
print(f"Segments needing nearest-centroid fallback: {len(unmatched)}")

if len(unmatched) > 0:
    nearest = gpd.sjoin_nearest(unmatched, landuse_centroids[[LANDUSE_FIELD, "geometry"]])
    nearest = (
        nearest.drop_duplicates(subset=SEG_ID_COL)[[SEG_ID_COL, LANDUSE_FIELD]]
        .rename(columns={LANDUSE_FIELD: "landuse"})
    )
    lookup = pd.concat([majority, nearest], ignore_index=True)
else:
    lookup = majority

print(f"Total lookup rows: {len(lookup)} (should equal footprint segment count: {len(segments_in_footprint)})")

Segments needing nearest-centroid fallback: 86297
Total lookup rows: 225247 (should equal footprint segment count: 225247)


## 7 · Coverage check — footprint segments

In [8]:
n_total_footprint = len(segments_in_footprint)
n_matched = lookup[SEG_ID_COL].nunique()
n_unmatched = n_total_footprint - n_matched

print(f"Footprint segments with no landuse match: {n_unmatched} ({100*n_unmatched/n_total_footprint:.2f}%)")
print(f"Previous (nearest-centroid) null rate was 33.2% — compare against that above.\n")
print(f"Class distribution (footprint segments):")
print(lookup["landuse"].value_counts(dropna=False).sort_index())

lookup.to_csv(config.LANDUSE_JOINED_CSV, index=False)
print(f"\nSaved lookup: {config.LANDUSE_JOINED_CSV}")

Footprint segments with no landuse match: 0 (0.00%)
Previous (nearest-centroid) null rate was 33.2% — compare against that above.

Class distribution (footprint segments):
landuse
1     110320
2      13198
3      12342
4       6032
5      15429
6       6520
7       3750
8       5903
9      25234
10     26519
Name: count, dtype: int64

Saved lookup: c:\dev\sdi-tehran\data/outputs/experiments/landuse-join-fix\landuse_joined.csv


## 8 · Merge into training set (non-spatial, keyed on Depthmap_R)

Geometry is dropped entirely from this point — the merge is a plain attribute join, matching how `training_set.csv` has always related to `segments.geojson`. `training_set` was already loaded in Section 1 (needed early for the footprint restriction in Section 4) — reused here, not reloaded.

In [9]:
old_null_pct = None
if "landuse" in training_set.columns:
    old_null_pct = 100 * training_set["landuse"].isna().sum() / len(training_set)

training_set_fixed = training_set.drop(columns=["landuse"], errors="ignore").merge(
    lookup, on=SEG_ID_COL, how="left"
)

print(f"Original training set: {len(training_set)} rows")
print(f"Fixed training set:    {len(training_set_fixed)} rows")
if old_null_pct is not None:
    print(f"Old landuse null rate: {old_null_pct:.2f}%")
new_null_pct = 100 * training_set_fixed["landuse"].isna().sum() / len(training_set_fixed)
print(f"New landuse null rate: {new_null_pct:.2f}%")
print(f"\nNew class distribution (training set only):")
print(training_set_fixed["landuse"].value_counts(dropna=False).sort_index())

training_set_fixed.to_csv(config.TRAINING_SET_FIXED_CSV, index=False)
print(f"\nSaved: {config.TRAINING_SET_FIXED_CSV}")

Original training set: 147414 rows
Fixed training set:    147414 rows
Old landuse null rate: 33.19%
New landuse null rate: 0.00%

New class distribution (training set only):
landuse
1     67669
2      6375
3      7141
4      3517
5      9088
6      4936
7      1494
8      4675
9     18727
10    23792
Name: count, dtype: int64

Saved: c:\dev\sdi-tehran\data/outputs/experiments/landuse-join-fix\training_set_landuse_fixed.csv


## 9 · Sanity checks before handing off to NB02

- Row count of `training_set_fixed` should exactly equal `training_set` (left join on an existing table — no rows should be added or dropped, only `landuse` values change).
- New null rate should be materially lower than 33.2% — in principle near-zero, since Section 4 guarantees every `training_set` segment enters the footprint and Section 6's fallback guarantees every footprint segment gets a label.
- Class distribution should now show exactly classes 1–10 (class 0 intentionally excluded) — if any of 1–10 vanished entirely, or if 0 reappears, that's worth investigating before proceeding.

In [10]:
assert len(training_set_fixed) == len(training_set), "Row count changed — merge introduced duplicates or drops!"
print("Row count check passed: no rows added or dropped.")

expected_classes = set(range(1, 11))  # 1 through 10 (0 intentionally excluded)
present_classes = set(training_set_fixed["landuse"].dropna().astype(int).unique())

missing_classes = expected_classes - present_classes
unexpected_classes = present_classes - expected_classes

if missing_classes:
    print(f"WARNING — expected classes missing from fixed training set: {sorted(missing_classes)}")
else:
    print("All 10 expected classes (1–10) present.")

if unexpected_classes:
    print(f"WARNING — unexpected classes present (should be empty, class 0 should have been excluded): {sorted(unexpected_classes)}")
else:
    print("No unexpected classes (class 0 successfully excluded).")

Row count check passed: no rows added or dropped.
All 10 expected classes (1–10) present.
No unexpected classes (class 0 successfully excluded).


In [11]:
# does the fallback subset skew differently than the buffer/vote subset?
fallback_dist = nearest["landuse"].value_counts(normalize=True).sort_index()
voted_dist = majority["landuse"].value_counts(normalize=True).sort_index()
print("Buffer/vote distribution:\n", voted_dist)
print("\nNearest-centroid fallback distribution:\n", fallback_dist)

Buffer/vote distribution:
 landuse
1     0.700058
2     0.049896
3     0.043051
4     0.014408
5     0.088967
6     0.014617
7     0.023404
8     0.011155
9     0.044174
10    0.010270
Name: proportion, dtype: float64

Nearest-centroid fallback distribution:
 landuse
1     0.151187
2     0.072598
3     0.073699
4     0.046699
5     0.035540
6     0.052018
7     0.005771
8     0.050442
9     0.221282
10    0.290763
Name: proportion, dtype: float64
